# Gundam NVIDIA Newton Simulation (Video Output)

This notebook sets up the NVIDIA Newton physics engine, loads the Gundam URDF, plays back the 'walk-forward' sample motion provided in the repository, and renders the output directly to an `.mp4` video saved to your Google Drive using Newton's headless EGL/GL Viewer.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install EGL dependencies for headless rendering in Colab
!apt-get update -y
!apt-get install -y libgl1-mesa-glx xvfb libxrender1
!pip install "newton[examples]" pandas imageio imageio-ffmpeg

In [ ]:
!rm -rf /content/gundam_robot
!git clone https://github.com/gundam-global-challenge/gundam_robot.git /content/gundam_robot
!sed -i 's/damping="3e2" friction="1e3"/damping="0.0" friction="0.0"/g' /content/gundam_robot/gundam_rx78_description/urdf/GGC_TestModel_rx78_20170112.urdf
# Replace package:// path with absolute path so Newton can find the meshes
!sed -i 's|package://gundam_rx78_description|/content/gundam_robot/gundam_rx78_description|g' /content/gundam_robot/gundam_rx78_description/urdf/GGC_TestModel_rx78_20170112.urdf

In [ ]:
import os
# Start virtual framebuffer to allow headless GL rendering
os.system('/usr/bin/Xvfb :99 -screen 0 1024x768x24 &')
os.environ['DISPLAY'] = ':99'

import newton
import warp as wp
import numpy as np
import pandas as pd
import imageio
from newton.solvers import SolverMuJoCo
from newton.viewer import ViewerGL

wp.init()

# Load the sample CSV motion
csv_path = "/content/gundam_robot/gundam_rx78_control/sample/csv/walk-forward.csv"
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

# Extract time and normalize to 0
time_col = df['time'].values
end_time = time_col[-1] - time_col[0]
time_col = time_col - time_col[0]

# Build Newton Model from URDF
print("Parsing URDF and building model...")
builder = newton.ModelBuilder()
builder.add_urdf(source="/content/gundam_robot/gundam_rx78_description/urdf/GGC_TestModel_rx78_20170112.urdf", ignore_inertial_definitions=True, floating=False)
model = builder.finalize()

state = model.state()

# Map CSV columns to Newton joint DOFs
joint_names = builder.joint_label
csv_to_newton_idx = {}
for csv_joint in df.columns:
    if csv_joint != 'time':
        try:
            idx = joint_names.index(csv_joint)
            dof_start = builder.joint_q_start[idx]
            csv_to_newton_idx[csv_joint] = dof_start
        except ValueError:
            pass

# Pre-extract joint target arrays for fast interpolation
joints_list = list(csv_to_newton_idx.keys())
dof_indices = [csv_to_newton_idx[j] for j in joints_list]
joint_data = np.radians(df[joints_list].values)

# Setup GL Viewer in Headless mode
fps = 60
viewer = ViewerGL(headless=True, width=640, height=480)
viewer.set_model(model)
viewer.set_camera(pos=(15.0, -10.0, 10.0), pitch=15.5, yaw=146.3)

print(f"Starting simulation and rendering {end_time:.2f} seconds of motion...")
sim_dt = 1.0 / 200.0
render_dt = 1.0 / float(fps)
render_accum = 0.0
current_time = 0.0

frames = []

while current_time <= end_time:
    # Interpolate target positions from CSV over continuous time
    idx1 = np.searchsorted(time_col, current_time)
    if idx1 == 0:
        interp_positions = joint_data[0]
    elif idx1 >= len(time_col):
        interp_positions = joint_data[-1]
    else:
        idx0 = idx1 - 1
        t0, t1 = time_col[idx0], time_col[idx1]
        alpha = (current_time - t0) / (t1 - t0) if t1 > t0 else 0.0
        interp_positions = (1.0 - alpha) * joint_data[idx0] + alpha * joint_data[idx1]
    
    target_positions = state.joint_q.numpy()
    for i, dof_idx in enumerate(dof_indices):
        target_positions[dof_idx] = interp_positions[i]
    
    state.joint_q = wp.array(target_positions, dtype=wp.float32)
    
    # Update forward kinematics so body transforms match joint targets
    newton.eval_fk(model, state.joint_q, state.joint_qd, state)
    
    render_accum += sim_dt
    if render_accum >= render_dt:
        viewer.begin_frame(time=current_time)
        viewer.log_state(state=state)
        viewer.end_frame()
        
        # Capture RGB array from OpenGL context
        img_wp = viewer.get_frame()
        frames.append(img_wp.numpy())
        render_accum -= render_dt # properly decrement instead of resetting to 0

    current_time += sim_dt
    
viewer.close()

# Save Video to Google Drive
video_path = "/content/drive/MyDrive/gundam_newton_motion.mp4"
if len(frames) > 0:
    imageio.mimwrite(video_path, frames, fps=fps, macro_block_size=None)
    print(f"Video saved to {video_path}")
else:
    print("No frames were captured.")
